# Base de datos: Campañas de marketing directo (banca portuguesa)

## Integrantes:

    -   
    -   

# Introducción (aprendizaje no supervisado)

En este trabajo abordamos el conjunto de datos de campañas de marketing directo de una entidad bancaria portuguesa desde una **perspectiva no supervisada**. A diferencia de la clasificación tradicional —que utiliza la variable objetivo `subscribed`— aquí buscamos **descubrir estructuras latentes** en los datos sin recurrir a etiquetas: segmentar clientes, identificar patrones de contacto y reconocer perfiles con comportamientos similares que puedan guiar decisiones comerciales.

El objetivo principal es **construir segmentos accionables** que apoyen la planificación de campañas: qué perfiles conviene contactar, por qué canal y en qué momento, y cuáles muestran señales de baja respuesta. Para ello, combinamos atributos **demográficos** (`age`, `job`, `education`), de **historial de interacción** (`campaign`, `pdays`, `previous`, `poutcome`) y de **contexto macroeconómico** (`emp.var.rate`, `cons.price.idx`, `euribor3m`, `nr.employed`), evitando el uso de `duration` (porque induce fuga de información) y sin utilizar `subscribed` en el proceso de descubrimiento (solo podrá emplearse luego para **validación ex post** de los segmentos).

Dado que el conjunto mezcla **variables numéricas y categóricas**, el preprocesamiento es clave: tratamiento de valores `unknown`, codificación apropiada de categorías, y **escalado** de variables continuas. La exploración incluirá técnicas de **reducción de dimensionalidad** para visualizar la estructura global y algoritmos de **clustering** con evaluación interna y análisis de **interpretabilidad** (perfiles medios por cluster, importancia de rasgos).

El resultado esperado es un conjunto de **segmentos interpretables**, alineados con la realidad del negocio, que permitan diseñar estrategias diferenciadas de contacto y oferta, optimizando recursos y mejorando la eficacia de las campañas sin depender inicialmente de una etiqueta de éxito.


## Descripción general
Conjunto de datos de campañas de **marketing directo vía llamadas telefónicas** de una institución bancaria en Portugal. El objetivo típico es predecir si un cliente **suscribirá un depósito a plazo**.

- **Observación:** un contacto (cliente × campaña).
- **Variable ilustrativa:** `subscribed` (sí/no).

> **Importante (fuga de información):** la variable `duration` solo se conoce **durante/después** de la llamada y está fuertemente correlacionada con el resultado. Para un modelo **realista en producción**, **no** debe usarse como predictor; puede utilizarse solo para **benchmarks**.

---

## Diccionario de datos

| # | Variable | Tipo | Descripción | Valores posibles |
|---:|---|---|---|---|
| 1 | `age` | numérico | Edad del cliente. | — |
| 2 | `job` | categórica | Tipo de empleo. | `admin.`, `blue-collar`, `entrepreneur`, `housemaid`, `management`, `retired`, `self-employed`, `services`, `student`, `technician`, `unemployed`, `unknown` |
| 3 | `marital` | categórica | Estado civil (*divorced* incluye divorciado/viudo). | `divorced`, `married`, `single`, `unknown` |
| 4 | `education` | categórica | Nivel educativo. | `basic.4y`, `basic.6y`, `basic.9y`, `high.school`, `illiterate`, `professional.course`, `university.degree`, `unknown` |
| 5 | `default` | categórica | ¿Crédito en mora/incumplimiento? | `no`, `yes`, `unknown` |
| 6 | `housing` | categórica | ¿Crédito hipotecario? | `no`, `yes`, `unknown` |
| 7 | `loan` | categórica | ¿Préstamo personal? | `no`, `yes`, `unknown` |
| 8 | `contact` | categórica | Canal del último contacto de la campaña actual. | `cellular`, `telephone` |
| 9 | `month` | categórica | Mes del último contacto. | `jan`, `feb`, `mar`, …, `nov`, `dec` |
|10 | `day_of_week` | categórica | Día de la semana del último contacto. | `mon`, `tue`, `wed`, `thu`, `fri` |
|11 | `duration` | numérico | Duración del último contacto (segundos). **Ver nota de fuga.** | — |
|12 | `campaign` | numérico | # de contactos en **esta** campaña (incluye el último). | — |
|13 | `pdays` | numérico | Días desde el último contacto en campañas previas (`999` = nunca contactado). | — |
|14 | `previous` | numérico | # de contactos **previos** a esta campaña. | — |
|15 | `poutcome` | categórica | Resultado de la campaña previa. | `failure`, `nonexistent`, `success` |
|16 | `emp.var.rate` | numérico | Tasa de variación del empleo (trimestral). | — |
|17 | `cons.price.idx` | numérico | Índice de precios al consumidor (mensual). | — |
|18 | `cons.conf.idx` | numérico | Índice de confianza del consumidor (mensual). | — |
|19 | `euribor3m` | numérico | Tasa Euribor a 3 meses (diaria). | — |
|20 | `nr.employed` | numérico | Número de empleados (trimestral). | — |
|21 | `subscribed` | binaria (objetivo) | ¿Suscribió depósito a plazo? | `yes`, `no` |

---

## Notas de modelado
- **Predictores recomendados:** variables demográficas (`age`, `job`, `education`, etc.), historial de campañas (`campaign`, `pdays`, `previous`, `poutcome`) y contexto macro (`emp.var.rate`, `euribor3m`, etc.).
- **Evitar en producción:** `duration` (conduce a **target leakage**).
- **Codificación:** tratar `unknown` como categoría explícita; considerar *one-hot encoding* para variables categóricas.
- **Escalas:** normalizar/estandarizar predictores continuos si se usan modelos sensibles a escala.


In [5]:
import pandas as pd
data = pd.read_csv("bank_marketing_dataset.csv")
data = data.drop(columns=['subscribed'])  # Eliminar columna irrelevante

In [6]:
pd.set_option('display.max_columns', None)
data.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,duration,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,261,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,149,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0
2,37,services,married,high.school,no,yes,no,telephone,may,mon,226,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,151,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0
4,56,services,married,high.school,no,no,yes,telephone,may,mon,307,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0


# ✅ Criterios de evaluación – Aprendizaje no supervisado

1) **Análisis exploratorio completo y preparación de datos**  
Realiza un **EDA** que caracterice la distribución de variables, relaciones relevantes y posibles sesgos: incluye tendencias, dispersión, outliers, correlaciones/associaciones y calidad de datos (faltantes y categorías `unknown`). A partir de este diagnóstico, deja el dataset listo para agrupar: tratamiento de faltantes, codificación de categóricas y escalado de continuas, justificando cada decisión y evitando fugas de información (no usar `subscribed` ni `duration` como insumo del clustering).

2) **Proyección y visualización de la estructura**  
Emplea una técnica de reducción de dimensionalidad (p. ej., PCA o UMAP) para explorar la forma global de los datos y sugerir separaciones naturales. Presenta visualizaciones claras que faciliten interpretar la estructura latente.

3) **Modelado de clusters**  
Selecciona y ejecuta un algoritmo acorde al tipo de variables (p. ej., k-means/k-medoids, k-prototypes, DBSCAN o mezcla gaussiana). Define y justifica los hiperparámetros (incluido el número de clusters cuando aplique) y reporta la estabilidad/consistencia de la solución.

4) **Descripción de los clusters**  
Perfila cada segmento en términos de sus rasgos distintivos (demografía, historial de campañas y contexto). Entrega un resumen comparativo (tablas/gráficos) que muestre claramente cómo difieren los clusters. **Las acciones comerciales derivadas** deberán ser propuestas aparte por los estudiantes; aquí se califica la **calidad y claridad de la descripción**.


### Punto 1

### Punto 2

### Punto 3

### Punto 4